# RNAP Hack 2026 · Reto 2: News in 60 Seconds
**Equipo:** _Equipo 4_  ·  **Integrantes:** _Jaime Esquivel, Alberto Moreno, Jared Dávila, Randhu_

Reporte técnico (cuaderno principal). En la **Fase 1** se completan las secciones 1, 2 y 3; en la **Fase 2**, las secciones 4 y 5; en la **Fase 3**, la sección 6.

In [ ]:
# Ejecuta esta celda una vez (Colab). Si ya tienes las librerías, no pasa nada.
!pip -q install transformers sentencepiece rouge_score accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Problema y datos

**Reto elegido:** News in 60 Seconds (Radio Cumbres).
**Problema:** resumir una noticia en español en **120 palabras o menos** (unos 60 segundos al aire, a ~2 palabras por segundo), usando solo hechos del artículo.
**Métrica principal:** ROUGE (ROUGE-1, ROUGE-2, ROUGE-L) contra el resumen del periodista, y % de resúmenes con ≤ 120 palabras.
**Fuente de los datos:** XL-Sum (csebuetnlp/xlsum), subconjunto en español: artículos de BBC Mundo con un resumen escrito por un periodista.

In [ ]:
import json, re, time, random
import numpy as np, pandas as pd
from pathlib import Path

# Carpeta de Google Drive donde dejaste los 3 archivos .jsonl (NO se suben a GitHub)
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/rnap_data')

train = pd.read_json(DATA_DIR/'spanish_train.jsonl', lines=True)
val   = pd.read_json(DATA_DIR/'spanish_val.jsonl',   lines=True)
test  = pd.read_json(DATA_DIR/'spanish_test.jsonl',  lines=True)

print('train:', len(train), '| val:', len(val), '| test:', len(test))
print('Columnas:', list(train.columns))

Mounted at /content/drive
train: 38110 | val: 4763 | test: 4763
Columnas: ['id', 'url', 'title', 'summary', 'text']


In [ ]:
# Tamaño de cada partición y longitud típica (palabras)
resumen = pd.DataFrame({
    'artículos':        [len(train), len(val), len(test)],
    'palabras texto (mediana)':    [train.text.str.split().str.len().median(),
                                    val.text.str.split().str.len().median(),
                                    test.text.str.split().str.len().median()],
    'palabras resumen (mediana)':  [train.summary.str.split().str.len().median(),
                                    val.summary.str.split().str.len().median(),
                                    test.summary.str.split().str.len().median()],
}, index=['train', 'val', 'test'])
resumen

,artículos,palabras texto (mediana),palabras resumen (mediana)
train,38110,771.0,27.0
val,4763,769.0,27.0
test,4763,768.0,27.0


**División de datos.** XL-Sum ya trae la división en entrenamiento (38,110), validación (4,763) y prueba (4,763), hecha **antes de entrenar**. Reglas del equipo:
- *train*: ajustar el modelo base (vocabulario e IDF de TF-IDF).
- *val*: elegir parámetros del modelo base (longitud objetivo del resumen).
- *test*: **solo** para las métricas finales, una vez.

Por tiempo de cómputo, ambos modelos se evalúan sobre **la misma muestra aleatoria fija** del conjunto de prueba (semilla 42).

In [ ]:
SEED = 42
N_EVAL = 200   # sube a 500 si Colab alcanza

test_s = test.sample(N_EVAL, random_state=SEED).reset_index(drop=True)
val_s  = val.sample(200,    random_state=SEED).reset_index(drop=True)
print(len(test_s), 'artículos de prueba |', len(val_s), 'de validación')

200 artículos de prueba | 200 de validación


## 2. Representación del texto

**Modelo base: TF-IDF por oración.** Cada oración se representa como un vector TF-IDF y su puntaje es el peso promedio de sus palabras. TF-IDF da peso alto a palabras frecuentes en la noticia pero poco comunes en el corpus (nombres, cifras, temas) y peso bajo a palabras vacías ("de", "la", "que"). Así una oración con muchas palabras informativas puntúa más. Es una representación de las primeras etapas del curso: no necesita GPU, es rápida y es fácil de explicar.

**Modelo preentrenado (sin ajuste): mT5_multilingual_XLSum.** Usa el tokenizador SentencePiece del propio modelo (subpalabras) y embeddings aprendidos; se aplica tal como viene.

## 3. Modelo base y preentrenado sin ajuste

### 3.1 Métrica (ROUGE para español)
El tokenizador por defecto de `rouge_score` borra las letras con acento, así que usamos uno propio que conserva `á é í ó ú ñ`.

In [ ]:
from rouge_score import rouge_scorer

class EsTokenizer:
    def tokenize(self, text):
        return re.findall(r'\w+', text.lower())

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], tokenizer=EsTokenizer())

def rouge_eval(preds, refs):
    acc = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        for k in acc:
            acc[k].append(s[k].fmeasure)
    return {k: round(100*float(np.mean(v)), 2) for k, v in acc.items()}

def pct_within(preds, limit=120):
    return round(100*float(np.mean([len(p.split()) <= limit for p in preds])), 1)

### 3.2 Modelo base: resumen extractivo con TF-IDF
Se ajusta el vocabulario y el IDF con una muestra de *train*. Después se puntúa cada oración y se eligen las mejores sin pasar del límite de palabras.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

t0 = time.time()
tfidf = TfidfVectorizer(lowercase=True, max_features=50000, sublinear_tf=True, norm=None)
tfidf.fit(train['text'].sample(20000, random_state=SEED))   # "entrenamiento" del modelo base
base_fit_time = time.time() - t0
print(f'TF-IDF ajustado en {base_fit_time:.1f} s | vocabulario: {len(tfidf.vocabulary_)} palabras')

def split_sentences(text):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sents if len(s.split()) >= 4]

def extractive_summary(text, max_words=120):
    sents = split_sentences(text)
    if not sents:
        return ' '.join(text.split()[:max_words])
    X = tfidf.transform(sents)
    lengths = np.array([len(s.split()) for s in sents])
    scores = np.asarray(X.sum(axis=1)).ravel() / np.maximum(lengths, 1)   # peso TF-IDF promedio
    chosen, total = [], 0
    for i in np.argsort(-scores):
        if total + lengths[i] <= max_words:
            chosen.append(i)
            total += lengths[i]
    if not chosen:                                   # primera oración demasiado larga
        return ' '.join(sents[int(np.argmax(scores))].split()[:max_words])
    return ' '.join(sents[i] for i in sorted(chosen))   # en el orden original

# Ejemplo
ej = test_s.loc[0, 'text']
print(extractive_summary(ej))

TF-IDF ajustado en 21.2 s | vocabulario: 50000 palabras
En esta fecha declarada feriado, se sientan a la mesa para celebrar el Día de Acción de Gracias (ThanksgivingDay), compartiendo pavo horneado, salsa de arándanos, puré de manzana y pastel de calabaza, entre otros platos. La costumbre hace que los que comparten la comida digan en alto de qué están agradecidos. El presidente Abraham Lincoln oficializó la celebración de Acción de Gracias en 1863. El gobernante ordenó celebrar Acción de Gracias luego de una campaña de la poeta y editora Sarah Josepha Hale, que había leído sobre el banquete de los colonizadores y los nativos de 1621, según cuenta la Enciclopedia Británica. Desde entonces, casi todo EE.UU. celebra Acción de Gracias. ¿Ya conoces nuestro canal de YouTube?


**Elegir la longitud objetivo con validación** (nunca con *test*). Todas las opciones respetan el límite de 120 palabras.

In [ ]:
resultados_val = {}
for L in [30, 45, 60, 90, 120]:
    preds = [extractive_summary(t, max_words=L) for t in val_s['text']]
    resultados_val[L] = rouge_eval(preds, val_s['summary'])['rouge1']
print('ROUGE-1 en validación por longitud objetivo:', resultados_val)

BEST_L = max(resultados_val, key=resultados_val.get)
print('Longitud elegida para el modelo base:', BEST_L, 'palabras')

ROUGE-1 en validación por longitud objetivo: {30: 17.45, 45: 18.83, 60: 19.08, 90: 17.76, 120: 16.47}
Longitud elegida para el modelo base: 60 palabras


In [ ]:
# Resultados del modelo base sobre el conjunto de PRUEBA (mide también el tiempo)
t0 = time.time()
base_preds = [extractive_summary(t, max_words=BEST_L) for t in test_s['text']]
base_time = time.time() - t0

base_res = rouge_eval(base_preds, test_s['summary'])
base_res['% ≤120 palabras'] = pct_within(base_preds)
base_res['tiempo total (s)'] = round(base_time, 2)
base_res['s por artículo']   = round(base_time/len(test_s), 3)
print(base_res)

{'rouge1': 18.78, 'rouge2': 2.82, 'rougeL': 12.2, '% ≤120 palabras': 100.0, 'tiempo total (s)': 0.32, 's por artículo': 0.002}


### 3.3 Preentrenado sin ajuste: mT5_multilingual_XLSum
Modelo encoder-decoder (Transformer) ya entrenado para resumir en varios idiomas, incluido el español. Se usa **tal como viene**, sin entrenarlo más, con los parámetros de generación de su documentación. *En Colab activa GPU:* Entorno de ejecución › Cambiar tipo › GPU T4.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = 'csebuetnlp/mT5_multilingual_XLSum'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', device)

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device).eval()

clean = lambda x: re.sub(r'\s+', ' ', x.strip())

def mt5_summaries(texts, bs=8):
    out = []
    for i in range(0, len(texts), bs):
        batch = [clean(t) for t in texts[i:i+bs]]
        enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            ids = model.generate(**enc, max_length=84, num_beams=4, no_repeat_ngram_size=2)
        out += tok.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return out

t0 = time.time()
pre_preds = mt5_summaries(list(test_s['text']))
pre_time = time.time() - t0

pre_res = rouge_eval(pre_preds, test_s['summary'])
pre_res['% ≤120 palabras'] = pct_within(pre_preds)
pre_res['tiempo total (s)'] = round(pre_time, 2)
pre_res['s por artículo']   = round(pre_time/len(test_s), 3)
print(pre_res)

Dispositivo: cuda


config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.33GB            

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model.safetensors: downloading bytes:           |  0.00B            

{'rouge1': 30.11, 'rouge2': 11.34, 'rougeL': 22.98, '% ≤120 palabras': 100.0, 'tiempo total (s)': 83.68, 's por artículo': 0.418}


### 3.4 Tabla comparativa

In [ ]:
tabla = pd.DataFrame({
    f'Base: TF-IDF extractivo ({BEST_L} palabras)': base_res,
    'Preentrenado sin ajuste: mT5_multilingual_XLSum': pre_res,
}).T
tabla

,rouge1,rouge2,rougeL,% ≤120 palabras,tiempo total (s),s por artículo
Base: TF-IDF extractivo (60 palabras),18.78,2.82,12.20,100.0,0.32,0.002
Preentrenado sin ajuste: mT5_multilingual_XLSum,30.11,11.34,22.98,100.0,83.68,0.418


In [ ]:
# Un ejemplo lado a lado (útil para la conclusión y para Fase 2)
i = 3
print('REFERENCIA (periodista):\n', test_s.loc[i, 'summary'], '\n')
print('BASE TF-IDF:\n', base_preds[i], '\n')
print('mT5 sin ajuste:\n', pre_preds[i])

REFERENCIA (periodista):
 Un plato de arroz con pollo y salsa soya que se vende por menos de US$2 es la especialidad de Hong Kong Soya Sauce Chicken Rice and Noodle, un puesto ambulante en las calles de Singapur que acaba de recibir una estrella Michelin. 

BASE TF-IDF:
 La concurrencia al puesto de comida quedó en evidencia cuando nuestra entrevista telefónica estuvo varias veces interrumpida mientras Chan Hon Meng preparaba los pedidos. La Guía Michelin Singapur menciona 62 puestos de comida callejeros, incluyendo Hong Kong Soya Sauce Chicken Rice and Noodle con su distintiva estrella a la derecha. 

mT5 sin ajuste:
 El dueño de un puesto ambulante de comida en Hong Kong dijo a BBC Mundo que todavía no ha decidido dónde colgar una estrella Michelin.


### 3.5 Conclusión (Fase 1)

Ambos modelos se evaluaron sobre la misma muestra de 200 artículos del conjunto de prueba (semilla 42). La longitud del modelo base (60 palabras) se eligió con validación, no con prueba.

| Modelo | ROUGE-1 | ROUGE-2 | ROUGE-L | % ≤ 120 palabras | Tiempo total | s por artículo |
| --- | --- | --- | --- | --- | --- | --- |
| Base: TF-IDF extractivo (60 palabras) | 18.78 | 2.82 | 12.20 | 100 % | 0.32 s | 0.002 |
| mT5_multilingual_XLSum sin ajuste | 30.11 | 11.34 | 22.98 | 100 % | 83.68 s (GPU) | 0.418 |

- **Calidad:** el preentrenado supera al base en las tres métricas. En ROUGE-1 la diferencia es de 11.3 puntos (30.11 contra 18.78) y en ROUGE-2 es de casi cuatro veces (11.34 contra 2.82). Esto indica que el modelo base casi no coincide con el resumen del periodista en secuencias de dos palabras.
- **Velocidad:** el base es mucho más rápido, unas 260 veces (0.32 s contra 83.68 s para los 200 artículos). Aun así, 0.42 s por artículo con el mT5 en GPU es aceptable para un uso en la radio.
- **Límite de 120 palabras:** los dos modelos lo cumplen en el 100 % de los casos. El base lo garantiza por construcción y el mT5 porque genera como máximo 84 tokens.
- **Fallas del base:** elige oraciones sueltas del artículo y no las conecta. En el ejemplo de la sección 3.4 toma una oración sobre la entrevista telefónica y no menciona el platillo ni su precio, que es lo importante del texto.
- **Fallas del preentrenado:** puede cambiar un hecho al reescribirlo. En ese mismo ejemplo dice que el dueño todavía no decide dónde colgar la estrella Michelin, cuando el artículo dice que el puesto la recibió. Esto es justo el problema de datos inventados que preocupa a Radio Cumbres. Además, el mT5 solo lee los primeros 512 tokens y las noticias tienen una mediana de unas 770 palabras, por lo que es probable que no vea el final del artículo.
- **Siguiente paso (Fase 2):** ajustar el mT5 con los datos de entrenamiento, evaluar en prueba con las mismas métricas y agregar la revisión de cifras y nombres que no aparecen en el artículo para detectar datos inventados.

> **Nota (Fase 3):** el análisis de atención de la sección 6.2 mostró que la frase sobre dónde colgar la estrella sí está en el artículo (habla de la placa con la estrella). El error real del ejemplo es ubicar el puesto en Hong Kong, que es parte de su nombre; el puesto está en Singapur.

---

## 4. Solución con Transformer: mT5 ajustado + verificador de datos

**Opción elegida: ajuste fino** de un modelo preentrenado, más un paso de verificación sobre lo que genera.

La Fase 1 mostró que el mT5 preentrenado es mucho mejor que el TF-IDF (ROUGE-1 30.11 contra 18.78), pero también sus dos problemas para la radio: a veces escribe datos que no están en la noticia y solo lee los primeros 512 tokens. La solución tiene dos partes:

1. **Ajuste fino en español.** `mT5_multilingual_XLSum` se entrenó con 45 idiomas a la vez. Lo seguimos entrenando solo con noticias en español de *train* para especializarlo en el estilo de BBC Mundo.
2. **Verificador de cifras y nombres.** El beam search ya calcula 4 hipótesis; pedimos las 4 en lugar de solo la mejor. El verificador revisa que cada número y cada nombre propio del resumen aparezca en el artículo y se queda con el primer candidato que pase. Si ninguno pasa, elige el que tenga menos datos sin respaldo y lo **marca para revisión** antes de salir al aire.

**Uso de los datos (igual que en la Fase 1):** *train* para el ajuste fino y para la lista de palabras comunes del verificador; *val* para decidir (mejor punto del entrenamiento y si el ajuste conviene); *test* solo en la sección 5, con la misma muestra de 200 artículos (semilla 42).

*Tiempo en Colab con GPU T4:* la celda de entrenamiento imprime una estimación después de los primeros 10 pasos.

### 4.1 Verificador de cifras y nombres
Reglas simples y fáciles de explicar:
- **Cifras:** cada número del resumen (sin puntos ni comas: `1.500` = `1500`) debe aparecer en el artículo, en dígitos o, si es chico, con letra (`15` = `quince`).
- **Nombres:** cada palabra con mayúscula inicial debe aparecer en el artículo, sin importar acentos ni mayúsculas. Se aceptan formas equivalentes frecuentes (`Estados Unidos` = `EE.UU.`).
- Para no marcar palabras comunes que solo llevan mayúscula por iniciar oración (*Cualquier*, *Según*), se usa una lista de palabras que en *train* aparecen más veces en minúscula que con mayúscula.

El mismo código está en `resumidor.py`, que es el que usa la interfaz.

In [ ]:
import unicodedata
from collections import Counter

LIMITE_PALABRAS = 120        # 60 segundos al aire
PALABRAS_POR_SEGUNDO = 2.0   # ritmo de lectura que pide el reto

# Cifras escritas con letra que cuentan como respaldo de una cifra en dígitos
_NUM_PALABRA = {
    '1': ['uno', 'una', 'un', 'primer', 'primera', 'primero'],
    '2': ['dos'], '3': ['tres'], '4': ['cuatro'], '5': ['cinco'], '6': ['seis'],
    '7': ['siete'], '8': ['ocho'], '9': ['nueve'], '10': ['diez'], '11': ['once'],
    '12': ['doce'], '13': ['trece'], '14': ['catorce'], '15': ['quince'],
    '16': ['dieciseis'], '17': ['diecisiete'], '18': ['dieciocho'], '19': ['diecinueve'],
    '20': ['veinte'], '30': ['treinta'], '40': ['cuarenta'], '50': ['cincuenta'],
    '60': ['sesenta'], '70': ['setenta'], '80': ['ochenta'], '90': ['noventa'],
    '100': ['cien', 'ciento'], '1000': ['mil'],
}

# Formas equivalentes comunes en BBC Mundo (el artículo dice "EE.UU." y el resumen "Estados Unidos")
_EEUU = ['ee', 'eeuu', 'eua', 'estadounidense', 'estadounidenses']
_ALIAS = {
    'estados': _EEUU, 'unidos': _EEUU, 'ee': ['estados'], 'uu': ['unidos'], 'eeuu': ['estados'],
    'onu': ['naciones'], 'naciones': ['onu'], 'unidas': ['onu'],
    'us': ['dolar', 'dolares'],                       # "US$2" frente a "2 dólares"
}

# Palabras comunes que llevan mayúscula solo por ir al inicio de una oración
_COMUNES = set('''
el la los las lo un una unos unas al del de en con por para sin sobre entre desde hasta hacia
segun tras ante bajo contra durante mediante y e o u ni pero sino aunque porque pues que quien
quienes cual cuales como cuando donde cuanto cuanta cuantos cuantas este esta estos estas ese esa
esos esas aquel aquella aquellos aquellas esto eso ello el ella ellos ellas yo tu usted ustedes
nosotros nosotras se su sus mi mis nuestro nuestra nuestros nuestras hay es son fue fueron era
eran ha han habia sera seran ser estar esta estan tambien tampoco ya aun todavia mas menos muy
mucho mucha muchos muchas poco pocos pocas todo toda todos todas otro otra otros otras cada
algunos algunas alguno alguna algun ningun ninguna varios varias ambos asi si no solo hoy ayer
manana ahora entonces luego despues antes mientras incluso ademas casi cerca alli aqui ahi nunca
siempre quizas tal tan tanto dicho dicha primer primero primera ultimo ultima nuevo nueva nuevos
nuevas gran grandes mayor mejor peor hace tiene tienen puede pueden cuatro cinco seis siete ocho
nueve diez cientos miles millones
'''.split())


def construir_comunes(textos, min_frec=3):
    """Palabras que en train aparecen más veces en minúscula que con mayúscula inicial
    (p. ej. 'cualquier', 'basta'). Así no se marcan como nombres cuando inician oración."""
    minus, mayus = Counter(), Counter()
    for t in textos:
        for w in re.findall(r'\w+', t):
            (mayus if w[0].isupper() else minus)[sin_acentos(w.lower())] += 1
    return {w for w, n in minus.items() if n >= min_frec and n >= mayus[w] and not w.isdigit()}


def limpiar(texto):
    """Espacios y saltos de línea a un solo espacio (como en la documentación del modelo)."""
    return re.sub(r'\s+', ' ', str(texto).strip())


def sin_acentos(texto):
    return ''.join(c for c in unicodedata.normalize('NFD', texto)
                   if unicodedata.category(c) != 'Mn')


def contar_palabras(texto):
    return len(str(texto).split())


def segundos_al_aire(texto):
    return contar_palabras(texto) / PALABRAS_POR_SEGUNDO


def recortar_palabras(texto, limite=LIMITE_PALABRAS):
    """Garantiza el límite: corta en el último punto antes de pasar de `limite` palabras."""
    palabras = str(texto).split()
    if len(palabras) <= limite:
        return str(texto).strip()
    corto = ' '.join(palabras[:limite])
    fin = max(corto.rfind('.'), corto.rfind('!'), corto.rfind('?'))
    return corto[:fin + 1] if fin > 0 else corto


def _cifras(texto):
    """Cifras en dígitos, sin separadores: '1.500' y '1,500' -> '1500'."""
    return {re.sub(r'[.,]', '', m) for m in re.findall(r'\d+(?:[.,]\d+)*', texto)}


def _nombres(texto):
    """Palabras que empiezan con mayúscula (nombres, siglas, lugares)."""
    return re.findall(r'\b[A-ZÁÉÍÓÚÑÜ][\wáéíóúñüÁÉÍÓÚÑÜ]*', texto)


def datos_no_encontrados(resumen, articulo):
    """Cifras y nombres del resumen que NO aparecen en el artículo.
    Si la lista está vacía, el resumen pasa la verificación."""
    palabras_art = set(re.findall(r'\w+', sin_acentos(articulo.lower())))
    cifras_art = _cifras(articulo)
    faltan = []
    for c in sorted(_cifras(resumen)):
        if c not in cifras_art and not any(p in palabras_art for p in _NUM_PALABRA.get(c, [])):
            faltan.append(c)
    for n in _nombres(resumen):
        clave = sin_acentos(n.lower())
        if clave in palabras_art or clave in _COMUNES or n in faltan:
            continue
        if any(a in palabras_art for a in _ALIAS.get(clave, [])):
            continue
        faltan.append(n)
    return faltan


def elegir_candidato(candidatos, articulo):
    """Recorre los candidatos en el orden del beam search (el primero es el más probable)
    y devuelve el primero que pasa la verificación. Si ninguno pasa, devuelve el que
    tiene menos datos sin respaldo y lo marca para revisión humana."""
    revisados = [(c, datos_no_encontrados(c, articulo)) for c in candidatos]
    for i, (c, faltan) in enumerate(revisados):
        if not faltan:
            return {'resumen': recortar_palabras(c), 'indice': i,
                    'no_encontrados': [], 'aprobado': True}
    i = min(range(len(revisados)), key=lambda k: (len(revisados[k][1]), k))
    return {'resumen': recortar_palabras(revisados[i][0]), 'indice': i,
            'no_encontrados': revisados[i][1], 'aprobado': False}

In [ ]:
# Lista de palabras comunes aprendida de train (la misma muestra con la que se ajustó el TF-IDF)
t0 = time.time()
_COMUNES |= construir_comunes(train['text'].sample(20000, random_state=SEED))
print(f'Palabras comunes aprendidas de train: {len(_COMUNES):,} ({time.time()-t0:.0f} s)\n')

# Prueba rápida con un ejemplo inventado
art_demo = 'El alcalde Ramírez anunció el lunes que la ciudad plantará 15.000 árboles antes de diciembre.'
for r in ['El alcalde Ramírez anunció que la ciudad plantará 15.000 árboles.',
          'El alcalde Juan Ramírez prometió plantar 20.000 árboles en dos años.']:
    faltan = datos_no_encontrados(r, art_demo)
    print(('PASA      ' if not faltan else f'REVISAR {faltan} ') + '→ ' + r)

In [ ]:
# Calibración en VALIDACIÓN: ¿qué tan seguido el resumen del periodista trae datos que no están en el artículo?
nf_ref_val = [datos_no_encontrados(s, clean(t)) for s, t in zip(val_s['summary'], val_s['text'])]
pct_ref_val = 100 * np.mean([bool(f) for f in nf_ref_val])
print(f'Resúmenes del periodista (val) con algún dato que no está en el artículo: {pct_ref_val:.1f} %')
print('Datos más frecuentes:', Counter(x for f in nf_ref_val for x in f).most_common(10))

**Lectura.** Cuatro de cada diez resúmenes escritos por el periodista incluyen algún dato que no está en el cuerpo de la noticia: nombres completos (*Donald* Trump cuando el texto solo dice Trump), *Estados Unidos* en notas que nunca lo mencionan, fechas o siglas desarrolladas (*Buró Federal de Investigaciones* cuando el texto solo dice FBI). En XL-Sum el resumen es la entrada de la nota de BBC Mundo, que se quitó del texto, así que puede traer información propia. Esto tiene dos consecuencias:
1. Un modelo entrenado para parecerse a esas referencias **aprende a agregar datos externos**, así que el ajuste fino por sí solo no resuelve el problema de Radio Cumbres. Por eso el verificador es parte de la solución.
2. Un sistema que solo usa datos del artículo nunca va a coincidir del todo con la referencia; el verificador puede bajar un poco ROUGE y ese costo se reporta en la sección 5.

### 4.2 Candidatos del mT5 sin ajuste
Se piden los 4 mejores resúmenes del beam search (`num_return_sequences=4`) con los mismos parámetros de la Fase 1, así que el primero debe ser el mismo resumen de la Fase 1. Se generan para *val* y para *test* ahora, antes de liberar la memoria de la GPU para el entrenamiento. Los de *test* **no se evalúan** hasta la sección 5.

In [ ]:
NUM_CAND = 4

def candidatos(mdl, textos, n=NUM_CAND, bs=8):
    """n candidatos por artículo, en orden de probabilidad del beam search."""
    mdl.eval()
    out = []
    for i in range(0, len(textos), bs):
        lote = [clean(t) for t in textos[i:i+bs]]
        enc = tok(lote, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            ids = mdl.generate(**enc, max_length=84, num_beams=4, no_repeat_ngram_size=2,
                               num_return_sequences=n)
        dec = tok.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        out += [dec[j*n:(j+1)*n] for j in range(len(lote))]
    return out

if 'model' not in globals():   # por si se reinició la sesión después de la sección 3
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device).eval()

cand_pre_val = candidatos(model, list(val_s['text']))
t0 = time.time()
cand_pre_test = candidatos(model, list(test_s['text']))   # se guarda; se evalúa en la sección 5
t_pre_cand = time.time() - t0

igual = np.mean([c[0] == p for c, p in zip(cand_pre_test, pre_preds)])
print(f'El primer candidato coincide con el resumen de la Fase 1 en el {100*igual:.0f} % de los artículos')
print(f'Generar 4 candidatos para los {len(test_s)} artículos tomó {t_pre_cand:.1f} s')

### 4.3 Ajuste fino con *train*

| Parámetro | Valor | Por qué |
| --- | --- | --- |
| Datos | 8,000 artículos de *train*, 1 época | cabe en una sesión de Colab; `N_TRAIN` se puede subir |
| Lote efectivo | 16 (lote de 4 × 4 pasos de acumulación) | memoria de la T4 |
| Optimizador | Adafactor, lr = 1e-4, 50 pasos de calentamiento | el estándar para T5/mT5 y usa menos memoria que Adam; lr bajo porque el modelo ya sabe resumir y solo queremos especializarlo |
| Precisión | fp32 | el mT5 produce NaN en fp16 |
| Longitudes | 512 tokens de entrada, 84 de salida | las mismas de la generación |
| Selección | pérdida en *val* cada 100 pasos; se guarda el mejor punto | el paso 0 (sin ajuste) también compite, así que el ajuste no puede empeorar el modelo elegido |

La pérdida es la entropía cruzada por token del resumen del periodista (*teacher forcing*).

In [ ]:
import gc, json, shutil
from transformers.optimization import Adafactor

ENTRENAR   = True    # False = cargar el modelo ajustado que ya quedó en Drive (si se reinició la sesión)
N_TRAIN    = 8000    # artículos de train (1 época)
BATCH      = 4       # si sale "CUDA out of memory": BATCH = 2 y ACUM = 8
ACUM       = 4       # lote efectivo = BATCH × ACUM = 16
LR         = 1e-4
WARMUP     = 50
MAX_IN, MAX_OUT = 512, 84
EVAL_CADA  = 100     # pasos de optimizador entre evaluaciones en val
PASOS_TOT  = N_TRAIN // (BATCH * ACUM)

TRABAJO   = Path('/content')
BEST_DIR  = TRABAJO / 'mt5_mejor'                  # disco local de Colab (rápido)
FT_DIR    = DATA_DIR / 'mt5_radio_cumbres'         # copia final en Drive, la usa la interfaz
HIST_PATH = DATA_DIR / 'historial_ajuste.json'

def preparar_lote(df):
    enc = tok([clean(t) for t in df['text']], max_length=MAX_IN, truncation=True,
              padding=True, return_tensors='pt')
    lab = tok(text_target=list(df['summary']), max_length=MAX_OUT, truncation=True,
              padding=True, return_tensors='pt').input_ids
    lab[lab == tok.pad_token_id] = -100            # el relleno no cuenta en la pérdida
    return enc.input_ids.to(device), enc.attention_mask.to(device), lab.to(device)

@torch.no_grad()
def perdida_val(mdl, df, bs=8):
    """Pérdida promedio por token en val; con ella se elige el mejor punto."""
    mdl.eval()
    total, n = 0.0, 0
    for i in range(0, len(df), bs):
        ids, am, lab = preparar_lote(df.iloc[i:i+bs])
        k = (lab != -100).sum().item()
        total += mdl(input_ids=ids, attention_mask=am, labels=lab).loss.item() * k
        n += k
    mdl.train()
    return total / n

tr = train.sample(N_TRAIN, random_state=SEED).reset_index(drop=True)
print(f'{len(tr):,} artículos de train | {PASOS_TOT} pasos de optimizador | evaluación en val cada {EVAL_CADA}')

In [ ]:
# Entrenamiento (con GPU T4 tarda del orden de media hora; se imprime una estimación al paso 10)
if 'model' in globals():          # libera el mT5 de la Fase 1: sus candidatos ya están guardados
    del model
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()

if ENTRENAR:
    torch.manual_seed(SEED)
    ft_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device)
    ft_model.config.use_cache = False
    opt = Adafactor(ft_model.parameters(), lr=LR, scale_parameter=False,
                    relative_step=False, warmup_init=False)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / WARMUP))

    v0 = perdida_val(ft_model, val_s)
    historial = {'paso': [], 'train': [], 'val_paso': [0], 'val': [v0], 'mejor_paso': 0}
    mejor = v0
    print(f'Paso 0 (sin ajuste) | pérdida val {v0:.4f}')

    ft_model.train()
    paso, acum, t0 = 0, 0.0, time.time()
    for k, i in enumerate(range(0, PASOS_TOT * BATCH * ACUM, BATCH)):
        ids, am, lab = preparar_lote(tr.iloc[i:i+BATCH])
        loss = ft_model(input_ids=ids, attention_mask=am, labels=lab).loss / ACUM
        loss.backward()
        acum += loss.item()
        if (k + 1) % ACUM:
            continue
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
        paso += 1
        historial['paso'].append(paso); historial['train'].append(acum); acum = 0.0
        if paso == 10:
            print(f'Tiempo estimado: {(time.time()-t0)/10*PASOS_TOT/60:.0f} min')
        if paso % EVAL_CADA == 0 or paso == PASOS_TOT:
            v = perdida_val(ft_model, val_s)
            historial['val_paso'].append(paso); historial['val'].append(v)
            marca = ''
            if v < mejor:
                mejor, historial['mejor_paso'] = v, paso
                ft_model.config.use_cache = True
                ft_model.save_pretrained(BEST_DIR); tok.save_pretrained(BEST_DIR)
                ft_model.config.use_cache = False
                marca = '  <- mejor hasta ahora, guardado'
            print(f'Paso {paso}/{PASOS_TOT} | pérdida train {np.mean(historial["train"][-EVAL_CADA:]):.4f} '
                  f'| val {v:.4f} | {(time.time()-t0)/60:.1f} min{marca}')
    historial['tiempo'] = time.time() - t0

    del opt, sched, ft_model
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()
    if historial['mejor_paso'] > 0:
        shutil.copytree(BEST_DIR, FT_DIR, dirs_exist_ok=True)
    HIST_PATH.write_text(json.dumps(historial))
else:
    historial = json.loads(HIST_PATH.read_text())

mejor_paso = historial['mejor_paso']
ft_model = AutoModelForSeq2SeqLM.from_pretrained(FT_DIR if mejor_paso > 0 else MODEL).to(device).eval()
ft_model.config.use_cache = True
print(f"\nMejor punto según val: paso {mejor_paso} de {PASOS_TOT} | entrenamiento: {historial['tiempo']/60:.1f} min")

In [ ]:
import matplotlib.pyplot as plt

GRIS, TINTA = '#8a8984', '#52514e'
def estilo(ax):
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)
    for s in ['left', 'bottom']:
        ax.spines[s].set_color('#c3c2b7')
    ax.tick_params(colors=TINTA, labelsize=9)
    ax.grid(axis='y', color='#e6e5e0', lw=0.8)
    ax.set_axisbelow(True)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(historial['paso'], pd.Series(historial['train']).rolling(20, min_periods=1).mean(),
        color='#2a78d6', lw=2, label='train (media móvil de 20 pasos)')
ax.plot(historial['val_paso'], historial['val'], color='#eb6834', lw=2, marker='o', ms=6, label='val')
if mejor_paso > 0:
    ax.axvline(mejor_paso, color=GRIS, ls='--', lw=1)
    ax.text(mejor_paso, ax.get_ylim()[1], ' mejor punto', color=TINTA, fontsize=9, va='top')
ax.set_xlabel('paso de optimizador', color=TINTA)
ax.set_ylabel('pérdida por token', color=TINTA)
ax.set_title('Gráfica 1. Curva de aprendizaje del ajuste fino', loc='left', fontsize=11)
estilo(ax); ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

### 4.4 Decisión con validación
**Regla fijada antes de ver *test*:** el modelo ajustado se usa solo si mejora ROUGE-1 en validación (el mismo criterio con el que se eligió la longitud del modelo base). El verificador va en el sistema final aunque baje un poco ROUGE, porque es un requisito de Radio Cumbres: solo hechos del artículo.

In [ ]:
def con_verificador(cands, textos):
    return [elegir_candidato(c, clean(t)) for c, t in zip(cands, textos)]

def pct_no_encontrado(preds, textos):
    return round(100 * float(np.mean([bool(datos_no_encontrados(p, clean(t)))
                                      for p, t in zip(preds, textos)])), 1)

cand_ft_val = candidatos(ft_model, list(val_s['text']))
filas = {}
for nombre, cands in [('mT5 sin ajuste', cand_pre_val), ('mT5 ajustado', cand_ft_val)]:
    sel = con_verificador(cands, val_s['text'])
    for etiqueta, preds in [(nombre, [c[0] for c in cands]),
                            (nombre + ' + verificador', [s['resumen'] for s in sel])]:
        fila = rouge_eval(preds, val_s['summary'])
        fila['% con dato no encontrado'] = pct_no_encontrado(preds, val_s['text'])
        filas[etiqueta] = fila
tabla_val = pd.DataFrame(filas).T

usar_ajustado = bool(mejor_paso > 0 and
                     tabla_val.loc['mT5 ajustado', 'rouge1'] > tabla_val.loc['mT5 sin ajuste', 'rouge1'])
FINAL = 'mT5 ajustado + verificador' if usar_ajustado else 'mT5 sin ajuste + verificador'
print('Decisión con ROUGE-1 en val:',
      'usar el modelo AJUSTADO' if usar_ajustado else 'el ajuste no mejoró; se usa el modelo SIN AJUSTE')
print('Sistema final:', FINAL)
tabla_val

### 4.5 Modelo para la interfaz
El modelo ajustado quedó en Drive (`rnap_data/mt5_radio_cumbres`). La interfaz (`app.py`) lo carga con la variable `MODEL_DIR`.

In [ ]:
MODELO_INTERFAZ = str(FT_DIR) if usar_ajustado else MODEL
print('MODEL_DIR para app.py:', MODELO_INTERFAZ)

# Opcional, para publicar la interfaz en Hugging Face Spaces (necesita un token de escritura):
# from huggingface_hub import login; login()
# ft_model.push_to_hub('USUARIO/mt5-radio-cumbres'); tok.push_to_hub('USUARIO/mt5-radio-cumbres')

## 5. Evaluación en prueba y análisis de errores

Ahora sí se usa *test*, **una vez**: la misma muestra de 200 artículos de la Fase 1 y las mismas métricas (ROUGE-1/2/L, % ≤ 120 palabras y tiempo), más una nueva:
- **% con dato no encontrado:** resúmenes con alguna cifra o nombre que no aparece en el artículo. En los sistemas con verificador son los que la interfaz marca para revisión.

Se comparan cinco sistemas para separar el efecto de cada parte: la línea base, el mT5 sin ajuste (Fase 1), el ajuste fino solo, el verificador solo y los dos juntos. En los sistemas con 4 candidatos el tiempo incluye generarlos; con beam search de 4 cuesta casi lo mismo que generar uno.

### 5.1 Resultados en prueba

In [ ]:
TEXTOS, REFS = list(test_s['text']), list(test_s['summary'])

t0 = time.time(); cand_ft_test = candidatos(ft_model, TEXTOS);          t_ft_cand = time.time() - t0
t0 = time.time(); sel_pre_test = con_verificador(cand_pre_test, TEXTOS); t_ver_pre = time.time() - t0
t0 = time.time(); sel_ft_test  = con_verificador(cand_ft_test, TEXTOS);  t_ver_ft  = time.time() - t0

BASE = f'Base: TF-IDF extractivo ({BEST_L} palabras)'
sistemas = {   # nombre: (resúmenes, tiempo total en segundos)
    BASE:                           (base_preds, base_time),
    'mT5 sin ajuste':               (pre_preds, pre_time),
    'mT5 sin ajuste + verificador': ([s['resumen'] for s in sel_pre_test], t_pre_cand + t_ver_pre),
    'mT5 ajustado':                 ([c[0] for c in cand_ft_test], t_ft_cand),
    'mT5 ajustado + verificador':   ([s['resumen'] for s in sel_ft_test], t_ft_cand + t_ver_ft),
}
filas = {}
for nombre, (preds, seg) in sistemas.items():
    fila = rouge_eval(preds, REFS)
    fila['% ≤120 palabras'] = pct_within(preds)
    fila['% con dato no encontrado'] = pct_no_encontrado(preds, TEXTOS)
    fila['tiempo total (s)'] = round(seg, 2)
    fila['s por artículo'] = round(seg / len(TEXTOS), 3)
    filas[nombre] = fila
tabla_test = pd.DataFrame(filas).T

pct_ref_test = pct_no_encontrado(REFS, TEXTOS)

# ROUGE-1 por artículo (para la gráfica 5 y el análisis de errores)
def r1_por_articulo(preds):
    return np.array([100 * scorer.score(r, p)['rouge1'].fmeasure for p, r in zip(preds, REFS)])
final_preds = sistemas[FINAL][0]
r1_base, r1_fin = r1_por_articulo(base_preds), r1_por_articulo(final_preds)
long_art = test_s['text'].str.split().str.len()
tercil = pd.qcut(long_art, 3, labels=['cortos', 'medianos', 'largos'])
r1_terc = pd.DataFrame({'base': r1_base, 'final': r1_fin, 'tercil': tercil}).groupby('tercil', observed=True).mean()

print(f'Resúmenes del periodista (test) con algún dato que no está en el artículo: {pct_ref_test} %')
print(f'Sistema final: {FINAL}')
tabla_test

### 5.2 Gráficas

In [ ]:
COL = {BASE: GRIS, 'mT5 sin ajuste': '#2a78d6', 'mT5 sin ajuste + verificador': '#eb6834',
       'mT5 ajustado': '#1baf7a', 'mT5 ajustado + verificador': '#eda100'}
nombres = list(sistemas)

fig, axs = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [1.5, 1]})
# Gráfica 2: ROUGE por sistema
ax, w = axs[0], 0.16
metricas = ['rouge1', 'rouge2', 'rougeL']
for j, n in enumerate(nombres):
    x = np.arange(3) + (j - 2) * w
    vals = tabla_test.loc[n, metricas].astype(float).values
    ax.bar(x, vals, w, color=COL[n], edgecolor='white', linewidth=1, label=n)
    for xi, v in zip(x, vals):
        ax.text(xi, v + 0.4, f'{v:.1f}', ha='center', fontsize=7, color=TINTA)
ax.set_xticks(np.arange(3), ['ROUGE-1', 'ROUGE-2', 'ROUGE-L'])
ax.set_ylabel('F1 (%)', color=TINTA)
ax.set_title('Gráfica 2. ROUGE en prueba (200 artículos)', loc='left', fontsize=11)
estilo(ax); ax.legend(frameon=False, fontsize=8, ncol=2, loc='upper right')
ax.set_ylim(0, tabla_test['rouge1'].astype(float).max() * 1.35)

# Gráfica 3: datos sin respaldo
ax = axs[1]
vals = tabla_test.loc[nombres, '% con dato no encontrado'].astype(float).values
y = np.arange(len(nombres))[::-1]
ax.barh(y, vals, 0.6, color=[COL[n] for n in nombres], edgecolor='white', linewidth=1)
for yi, v in zip(y, vals):
    ax.text(v + 0.5, yi, f'{v:.1f} %', va='center', fontsize=8, color=TINTA)
ax.axvline(pct_ref_test, color=TINTA, ls='--', lw=1)
ax.text(pct_ref_test, -0.85, f' referencia del periodista ({pct_ref_test:.1f} %)', fontsize=8, color=TINTA, va='center')
ax.set_ylim(-1.2, len(nombres) - 0.5)
ax.set_yticks(y, nombres, fontsize=8)
ax.set_xlabel('% de resúmenes con alguna cifra o nombre que no está en el artículo', color=TINTA, fontsize=9)
ax.set_title('Gráfica 3. Datos sin respaldo en el artículo', loc='left', fontsize=11)
estilo(ax); ax.grid(axis='y', visible=False); ax.grid(axis='x', color='#e6e5e0', lw=0.8)
ax.set_xlim(0, max(vals.max(), pct_ref_test) * 1.25 + 1)
plt.tight_layout(); plt.show()

In [ ]:
import inspect
fig, axs = plt.subplots(1, 2, figsize=(13, 4))

# Gráfica 4: longitud de los resúmenes
ax = axs[0]
grupos = {'Referencia (periodista)': REFS, BASE: base_preds, 'mT5 sin ajuste': pre_preds, FINAL + ' (final)': final_preds}
largos = [[len(p.split()) for p in v] for v in grupos.values()]
horiz = ({'orientation': 'horizontal'} if 'orientation' in inspect.signature(ax.boxplot).parameters
         else {'vert': False})
bp = ax.boxplot(largos, **horiz, widths=0.5, patch_artist=True, showfliers=True,
                medianprops={'color': 'white', 'lw': 2}, flierprops={'ms': 4, 'mec': GRIS},
                whiskerprops={'color': TINTA}, capprops={'color': TINTA})
for caja, c in zip(bp['boxes'], [TINTA, GRIS, '#2a78d6', COL[FINAL]]):
    caja.set_facecolor(c); caja.set_edgecolor(c)
ax.axvline(120, color='#e34948', lw=1.5)
ax.text(120, len(grupos) + 0.35, ' límite: 120 palabras (60 s)', color='#e34948', fontsize=8)
ax.set_yticks(range(1, len(grupos) + 1), list(grupos), fontsize=8)
ax.set_xlabel('palabras por resumen', color=TINTA)
ax.set_title('Gráfica 4. Longitud de los resúmenes', loc='left', fontsize=11)
estilo(ax); ax.grid(axis='y', visible=False); ax.grid(axis='x', color='#e6e5e0', lw=0.8)

# Gráfica 5: ROUGE-1 según la longitud del artículo (el mT5 solo lee 512 tokens)
ax = axs[1]
rangos = [f'{n}\n({long_art[tercil == n].min()}–{long_art[tercil == n].max()} palabras)' for n in ['cortos', 'medianos', 'largos']]
x = np.arange(3)
for j, (col, n, c) in enumerate([('base', BASE, GRIS), ('final', FINAL + ' (final)', COL[FINAL])]):
    ax.bar(x + (j - 0.5) * 0.3, r1_terc[col].values, 0.3, color=c, edgecolor='white', linewidth=1, label=n)
    for xi, v in zip(x + (j - 0.5) * 0.3, r1_terc[col].values):
        ax.text(xi, v + 0.4, f'{v:.1f}', ha='center', fontsize=8, color=TINTA)
ax.set_xticks(x, rangos, fontsize=8)
ax.set_ylabel('ROUGE-1 promedio', color=TINTA)
ax.set_title('Gráfica 5. ROUGE-1 según la longitud del artículo', loc='left', fontsize=11)
estilo(ax); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

**Cómo leer las gráficas.** La gráfica 2 compara la calidad contra el resumen del periodista; la 3 mide el problema que más preocupa a Radio Cumbres (datos que no vienen en la noticia), con la línea punteada como referencia de qué tanto lo hace el propio periodista; la 4 muestra que todos los sistemas quedan lejos del límite de 120 palabras; y la 5 revisa si los artículos largos, que el mT5 no alcanza a leer completos, salen peor.

### 5.3 Análisis de errores
Se buscan automáticamente en *test* los siguientes tipos de error del sistema final y se muestra el peor caso (menor ROUGE-1) de cada uno, sin repetir artículos:

| Tipo | Por qué importa para la radio |
| --- | --- |
| **Cifra sin respaldo** | Un número equivocado al aire (muertos, montos, fechas) es el error más grave. |
| **Nombre sin respaldo** | Atribuir algo a la persona o al lugar equivocado. |
| **Error corregido por el verificador** | Muestra qué inventaba el candidato más probable y qué eligió el verificador en su lugar. |
| **Información clave después del corte** | El mT5 solo lee 512 tokens; si lo importante está al final, el resumen habla de otra cosa. |
| **Pasa la verificación pero falla el enfoque** | Todas las cifras y nombres existen, pero el resumen elige un hecho secundario o relaciona mal los datos. El verificador no puede verlo. |
| **Resumen muy corto o en forma de pregunta** | Sirve poco como nota de 60 segundos. |

Primero se cuenta qué tan frecuente es cada tipo en los 200 artículos y después se muestran los casos.

In [ ]:
from IPython.display import Markdown, display

sel_final  = sel_ft_test if usar_ajustado else sel_pre_test
cand_final = cand_ft_test if usar_ajustado else cand_pre_test

def palabras(texto):
    return re.findall(r'\w+', sin_acentos(clean(texto).lower()))

def leidas_y_total(texto):
    """Palabras que alcanza a leer el mT5 (512 tokens) y palabras totales del artículo."""
    ids = tok(clean(texto), truncation=False).input_ids
    return len(palabras(tok.decode(ids[:512], skip_special_tokens=True))), len(palabras(texto))

def ref_despues_del_corte(ref, texto, leidas):
    """De las palabras de contenido de la referencia que sí están en el artículo,
    qué fracción aparece por primera vez después de lo que leyó el modelo."""
    primera = {}
    for k, w in enumerate(palabras(texto)):
        primera.setdefault(w, k)
    cont = [w for w in set(palabras(ref)) if len(w) >= 5 and w in primera]
    return float(np.mean([primera[w] >= leidas for w in cont])) if cont else 0.0

filas = []
for i in range(len(TEXTOS)):
    leidas, total = leidas_y_total(TEXTOS[i])
    s = sel_final[i]
    filas.append({'i': i, 'r1': r1_fin[i], 'faltan': s['no_encontrados'], 'aprobado': s['aprobado'],
                  'indice': s['indice'], 'leidas': leidas, 'total': total,
                  'despues': ref_despues_del_corte(REFS[i], TEXTOS[i], leidas),
                  'palabras': len(final_preds[i].split()),
                  'pregunta': final_preds[i].strip().endswith('?')})
info = pd.DataFrame(filas)

tipos = {
    'Cifra sin respaldo':                          info.faltan.apply(lambda f: any(x.isdigit() for x in f)),
    'Nombre sin respaldo':                         info.faltan.apply(lambda f: any(not x.isdigit() for x in f)),
    'Error corregido por el verificador':          (info.indice > 0) & info.aprobado,
    'Información clave después del corte':         (info.leidas < info.total) & (info.despues >= 0.5),
    'Pasa la verificación pero falla el enfoque':  info.aprobado & (info.r1 < 15),
    'Resumen muy corto o en forma de pregunta':    (info.palabras <= 12) | info.pregunta,
}
tabla_errores = pd.DataFrame({'artículos': {t: int(m.sum()) for t, m in tipos.items()},
                              '% de la muestra': {t: round(100 * m.mean(), 1) for t, m in tipos.items()}})
print(f'El mT5 lee el artículo completo solo en el {100*np.mean(info.leidas >= info.total):.0f} % de los casos; '
      f'en promedio alcanza el {100*np.mean(np.minimum(1, info.leidas/info.total)):.0f} % de las palabras.')
tabla_errores

In [ ]:
def diagnostico(f):
    i, partes = int(f.i), []
    if f.indice > 0:
        antes = datos_no_encontrados(cand_final[i][0], clean(TEXTOS[i]))
        partes.append(f'El candidato más probable traía datos que no están en el artículo ({", ".join(antes)}); '
                      f'el verificador eligió el candidato {f.indice + 1} de {NUM_CAND}.')
    if f.faltan:
        partes.append(f'El resumen final menciona **{", ".join(f.faltan)}**, que no aparece en el artículo. '
                      'Ninguno de los 4 candidatos pasó la verificación, así que la interfaz lo marcaría para revisión.')
    if f.leidas < f.total:
        partes.append(f'El modelo leyó {f.leidas} de {f.total} palabras ({100*f.leidas/f.total:.0f} %) y el '
                      f'{100*f.despues:.0f} % de las palabras clave de la referencia aparece después de ese corte.')
    else:
        partes.append('El modelo leyó el artículo completo.')
    if f.aprobado and f.r1 < 15:
        partes.append('Todas sus cifras y nombres están en el artículo, así que el verificador no puede detectar '
                      'el problema: el error está en qué hecho eligió o en cómo relacionó los datos.')
    if f.palabras <= 12 or f.pregunta:
        partes.append(f'Con {f.palabras} palabras (≈{f.palabras/2:.0f} s) no alcanza para una nota de radio.')
    return ' '.join(partes)

usados, casos = set(), []
def tomar(tipo, mascara):
    c = info[mascara & ~info.i.isin(usados)].sort_values('r1')
    if len(c):
        usados.add(int(c.iloc[0].i)); casos.append((tipo, c.iloc[0]))

for tipo, mascara in tipos.items():
    tomar(tipo, mascara)
while len(casos) < 6:                       # garantiza al menos 6 casos aunque falte algún tipo
    tomar('ROUGE-1 bajo (otro caso)', info.r1 >= 0)

for n, (tipo, f) in enumerate(casos, 1):
    i = int(f.i)
    texto = (f'#### Caso {n}. {tipo}\n'
             f'**Noticia:** {test_s.loc[i, "title"]} · ROUGE-1 = {f.r1:.1f} · {f.palabras} palabras\n\n'
             f'- **Referencia (periodista):** {REFS[i]}\n')
    if f.indice > 0:
        texto += f'- **Candidato más probable (antes del verificador):** {cand_final[i][0]}\n'
    texto += f'- **Sistema final:** {final_preds[i]}\n- **Diagnóstico:** {diagnostico(f)}\n'
    display(Markdown(texto))

**Lectura de los casos.** Los errores caen en tres familias, y cada una pide una solución distinta:
- **Datos inventados (cifras y nombres sin respaldo).** Vienen de cómo se entrenó el modelo: aprendió a imitar resúmenes que traen información externa. El verificador los atrapa cuando el dato no está en el artículo; si ninguno de los 4 candidatos está limpio, el resumen sale marcado para que el locutor lo corrija.
- **Errores de lectura (información después del corte).** No son culpa de la generación sino de la entrada: el modelo nunca vio esa parte de la noticia. Se atacan seleccionando las oraciones más importantes antes de pasar el texto al mT5.
- **Errores de enfoque o de relación.** El resumen usa datos reales del artículo pero elige un hecho secundario o los conecta mal (como el ejemplo de la Fase 1 sobre la estrella Michelin). El verificador no los ve; por eso la interfaz presenta el resumen como borrador para que una persona lo revise antes de salir al aire.

### 5.4 Conclusión (Fase 2)
La siguiente celda escribe la conclusión con los números de esta corrida, para que el texto y la tabla siempre coincidan.

In [ ]:
T = tabla_test.astype(float)
b, p, f = T.loc[BASE], T.loc['mT5 sin ajuste'], T.loc[FINAL]
top1 = T.loc['mT5 ajustado' if usar_ajustado else 'mT5 sin ajuste']
d = lambda x: f'{x:+.2f}'
v_pre, v_ft = tabla_val.loc['mT5 sin ajuste', 'rouge1'], tabla_val.loc['mT5 ajustado', 'rouge1']
cambio = 100 * np.mean([s['indice'] > 0 for s in sel_final])
med_fin, med_ref = int(np.median([len(x.split()) for x in final_preds])), int(np.median([len(x.split()) for x in REFS]))

if usar_ajustado:
    ajuste = (f'El ajuste con {N_TRAIN:,} artículos en español ({historial["tiempo"]/60:.0f} min de GPU) subió ROUGE-1 en '
              f'validación de {v_pre:.2f} a {v_ft:.2f}; en prueba el modelo ajustado sin verificador pasó de '
              f'{p.rouge1:.2f} a {T.loc["mT5 ajustado", "rouge1"]:.2f}. La mejor pérdida en val se alcanzó en el paso '
              f'{mejor_paso} de {PASOS_TOT} (gráfica 1).')
else:
    ajuste = (f'En validación el ajuste no superó al modelo original (ROUGE-1 {v_ft:.2f} contra {v_pre:.2f}), así que el '
              f'sistema final usa el mT5 sin ajuste. Una explicación probable es que `mT5_multilingual_XLSum` ya se '
              f'entrenó con estas mismas noticias de *train*, junto con las de otros 44 idiomas. Para no esconder el '
              f'resultado, en prueba el modelo ajustado obtuvo ROUGE-1 {T.loc["mT5 ajustado", "rouge1"]:.2f}.')

velocidad = (f'{f["s por artículo"]:.2f} s por artículo en {"GPU" if device == "cuda" else "CPU"} '
             f'(generar 4 candidatos y verificarlos), contra '
             f'{b["s por artículo"]:.3f} s del TF-IDF.' +
             (' Para la radio, tener el resumen en menos de un segundo es suficiente.' if f['s por artículo'] < 1 else ''))

display(Markdown(f"""
**Resultados en prueba** (200 artículos, semilla 42). El sistema final, **{FINAL}**, obtiene ROUGE-1 {f.rouge1:.2f},
ROUGE-2 {f.rouge2:.2f} y ROUGE-L {f.rougeL:.2f}. Frente a la línea base TF-IDF son {d(f.rouge1-b.rouge1)},
{d(f.rouge2-b.rouge2)} y {d(f.rougeL-b.rougeL)} puntos; frente al mT5 sin ajuste de la Fase 1, {d(f.rouge1-p.rouge1)},
{d(f.rouge2-p.rouge2)} y {d(f.rougeL-p.rougeL)}.

- **Ajuste fino.** {ajuste}
- **Datos sin respaldo.** El {top1['% con dato no encontrado']:.1f} % de los resúmenes del mT5 {'ajustado' if usar_ajustado else 'sin ajuste'}
  trae alguna cifra o nombre que no está en el artículo. Con el verificador baja a {f['% con dato no encontrado']:.1f} %, y esos
  resúmenes salen marcados para revisión. El verificador cambió de candidato en el {cambio:.0f} % de los artículos y su costo
  en ROUGE-1 fue de {d(f.rouge1 - top1.rouge1)} puntos. Como comparación, el {pct_ref_test:.1f} % de los resúmenes del periodista
  también trae datos que no están en el cuerpo de la noticia.
- **Velocidad.** {velocidad}
- **Límite de 120 palabras.** Lo cumple el {f['% ≤120 palabras']:.0f} % de los resúmenes. La mediana es de {med_fin} palabras
  (unos {med_fin/2:.0f} s al aire); la del periodista es de {med_ref} palabras.
- **Artículos largos.** En el tercio de artículos más largos el sistema final obtiene ROUGE-1 {r1_terc.loc['largos', 'final']:.1f},
  contra {r1_terc.loc['cortos', 'final']:.1f} en el tercio más corto (gráfica 5). El mT5 lee el artículo completo solo en el
  {100*np.mean(info.leidas >= info.total):.0f} % de los casos.
"""))

**Límites y siguientes pasos**
- **ROUGE no mide fidelidad.** Compara contra una sola referencia que, en cuatro de cada diez casos, trae datos que no están en la noticia. Por eso se agregó la métrica de datos sin respaldo y se acepta perder un poco de ROUGE a cambio de no inventar datos.
- **El verificador es estricto pero superficial.** Revisa que las cifras y los nombres existan en el artículo, no que estén bien relacionados, y marca como sospechosos datos ciertos que el artículo no menciona (por ejemplo, *Donald* en "Donald Trump"). Es una ayuda para el locutor, no un juez final.
- **Entrada truncada.** El mT5 solo lee 512 tokens. Un siguiente paso es elegir las oraciones más informativas del artículo completo (con el mismo TF-IDF de la línea base) antes de pasarlo al modelo.
- **Interfaz v1.** `app.py` usa exactamente este sistema: genera los 4 candidatos, aplica el verificador, muestra el resumen con su duración al aire y resalta los datos que no aparecen en la noticia.

### 5.5 Resumen para PROGRESS.md

In [ ]:
cols = ['rouge1', 'rouge2', 'rougeL', '% ≤120 palabras', '% con dato no encontrado', 's por artículo']
lineas = ['| Sistema (test, 200 artículos) | ' + ' | '.join(cols) + ' |', '| --- ' * (len(cols) + 1) + '|']
for n, r in tabla_test[cols].astype(float).iterrows():
    lineas.append(f'| {n}{" **(final)**" if n == FINAL else ""} | ' + ' | '.join(f'{v:g}' for v in r) + ' |')
print('\n'.join(lineas))
print(f'\nReferencia del periodista con datos fuera del artículo: {pct_ref_test} %')
print(f'Ajuste fino: {N_TRAIN} artículos, {PASOS_TOT} pasos, mejor paso {mejor_paso}, '
      f'{historial["tiempo"]/60:.0f} min | modelo para la interfaz: {MODELO_INTERFAZ}')

## 6. Comparación final y límites

### 6.1 Tabla final: línea base contra la solución con Transformer
Se comparan la línea base de la Fase 1, el mT5 sin ajuste y el sistema final, con los mismos 200 artículos de prueba. Para saber si la diferencia con la línea base es real o efecto de la muestra, se calcula un **intervalo de confianza del 95 % con bootstrap**: se remuestrean los 200 artículos 2,000 veces y se ve cuánto varía la diferencia promedio.

In [ ]:
METRICAS = ['rouge1', 'rouge2', 'rougeL']
NOMBRE_M = {'rouge1': 'ROUGE-1', 'rouge2': 'ROUGE-2', 'rougeL': 'ROUGE-L'}

def por_articulo(preds, m):
    return np.array([100 * scorer.score(r, p)[m].fmeasure for p, r in zip(preds, REFS)])

def bootstrap_ic(valores, n=2000, semilla=SEED):
    rng = np.random.default_rng(semilla)
    medias = [rng.choice(valores, len(valores)).mean() for _ in range(n)]
    return np.percentile(medias, [2.5, 97.5])

cols = METRICAS + ['% ≤120 palabras', '% con dato no encontrado', 's por artículo']
tabla_final = tabla_test.loc[[BASE, 'mT5 sin ajuste', FINAL], cols].astype(float)
tabla_final.loc['Diferencia: final − base'] = tabla_final.loc[FINAL] - tabla_final.loc[BASE]

pa = {n: {m: por_articulo(sistemas[n][0], m) for m in METRICAS} for n in [BASE, 'mT5 sin ajuste', FINAL]}
ic = pd.DataFrame({
    NOMBRE_M[m]: {
        'final − base': '{:+.2f} ({:+.2f} a {:+.2f})'.format(
            (pa[FINAL][m] - pa[BASE][m]).mean(), *bootstrap_ic(pa[FINAL][m] - pa[BASE][m])),
        'final − mT5 sin ajuste': '{:+.2f} ({:+.2f} a {:+.2f})'.format(
            (pa[FINAL][m] - pa['mT5 sin ajuste'][m]).mean(), *bootstrap_ic(pa[FINAL][m] - pa['mT5 sin ajuste'][m])),
    } for m in METRICAS})
display(tabla_final.round(3))
print('Diferencia promedio por artículo (intervalo de confianza del 95 %, bootstrap):')
display(ic)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
x, w = np.arange(3), 0.36
for j, (n, c, etiqueta) in enumerate([(BASE, GRIS, BASE), (FINAL, COL[FINAL], FINAL + ' (final)')]):
    medias = np.array([pa[n][m].mean() for m in METRICAS])
    lims = np.array([bootstrap_ic(pa[n][m]) for m in METRICAS])
    err = np.vstack([medias - lims[:, 0], lims[:, 1] - medias])
    xs = x + (j - 0.5) * w
    ax.bar(xs, medias, w, color=c, edgecolor='white', linewidth=1, label=etiqueta)
    ax.errorbar(xs, medias, yerr=err, fmt='none', ecolor=TINTA, elinewidth=1, capsize=3)
    for xi, v, hi in zip(xs, medias, lims[:, 1]):
        ax.text(xi, hi + 0.6, f'{v:.1f}', ha='center', fontsize=8, color=TINTA)
ax.set_xticks(x, [NOMBRE_M[m] for m in METRICAS])
ax.set_ylabel('F1 (%)', color=TINTA)
ax.set_title('Gráfica 6. Línea base contra solución final (barras: IC 95 %)', loc='left', fontsize=11)
ax.set_ylim(0, ax.get_ylim()[1] * 1.2)
estilo(ax); ax.legend(frameon=False, fontsize=8, loc='upper right')
plt.tight_layout(); plt.show()

### 6.2 El elemento de atención del mT5

El mT5 es un Transformer **codificador-decodificador**. La celda siguiente imprime su configuración real.

**Cómo funciona una cabeza de atención.** Cada token se convierte en tres vectores: consulta ($Q$), clave ($K$) y valor ($V$). El peso que un token le da a otro sale del producto punto de su consulta con la clave del otro, normalizado con softmax, y la salida es el promedio de los valores con esos pesos:

$$\text{Atención}(Q, K, V) = \text{softmax}\left(QK^\top + B\right)V$$

Dos detalles propios de T5/mT5: no divide entre $\sqrt{d_k}$ como el Transformer original (ese factor se absorbe en la inicialización de los pesos), y en lugar de embeddings de posición absolutos suma un **sesgo de posición relativa** $B$ en la autoatención, que depende solo de la distancia entre los dos tokens. Cada capa tiene varias **cabezas** en paralelo y cada una puede aprender una relación distinta; sus salidas se concatenan.

**Las tres atenciones del mT5**

| Dónde | Quién mira a quién | Para qué sirve al resumir |
| --- | --- | --- |
| Autoatención del codificador | cada token del artículo mira a todos los demás, en ambas direcciones | entender la noticia: conectar "estrella" con "Michelin" o un pronombre con su nombre |
| Autoatención enmascarada del decodificador | cada token del resumen mira solo a los que ya escribió | escribir español fluido, una palabra a la vez |
| **Atención cruzada** | cada token del resumen mira a todos los tokens del artículo | decidir de qué parte de la noticia sale cada palabra |

La atención cruzada es la que conecta el resumen con la noticia, así que es la que analizamos. **Costo:** la autoatención compara todos los pares de tokens, así que su memoria y su tiempo crecen con el cuadrado de la longitud. Por eso el modelo trabaja con 512 tokens y no con el artículo completo.

**Cómo la medimos.** Se le da al modelo el artículo y un resumen ya escrito (como en el entrenamiento) con `output_attentions=True`, y se promedia la atención cruzada de todas las capas y cabezas. Así queda una matriz *palabras del resumen × palabras del artículo*; los subtokens de SentencePiece se agrupan en palabras.

In [ ]:
cfg = ft_model.config
print(f'Capas: {cfg.num_layers} en el codificador + {cfg.num_decoder_layers} en el decodificador')
print(f'Cabezas de atención por capa: {cfg.num_heads} | dimensión por cabeza (d_kv): {cfg.d_kv} | d_model: {cfg.d_model}')
print(f'Posición relativa: {cfg.relative_attention_num_buckets} intervalos, hasta {cfg.relative_attention_max_distance} tokens de distancia')
print(f'Vocabulario SentencePiece: {cfg.vocab_size:,} | parámetros: {sum(p.numel() for p in ft_model.parameters())/1e6:.0f} millones')

# Para leer los pesos de atención hace falta la implementación "eager" (la rápida no los devuelve)
mt5_att = AutoModelForSeq2SeqLM.from_pretrained(MODEL, attn_implementation='eager').to(device).eval()
mt5_fin_att = (AutoModelForSeq2SeqLM.from_pretrained(str(FT_DIR), attn_implementation='eager').to(device).eval()
               if usar_ajustado else mt5_att)

def atencion_cruzada(mdl, texto, resumen):
    """Atención cruzada promediada en capas y cabezas: matriz (tokens del resumen × tokens del artículo)."""
    enc = tok(clean(texto), return_tensors='pt', truncation=True, max_length=512).to(device)
    lab = tok(text_target=resumen, return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        out = mdl(**enc, labels=lab, output_attentions=True)
    A = torch.stack(out.cross_attentions)[:, 0].mean(dim=(0, 1)).float().cpu().numpy()
    return A, tok.convert_ids_to_tokens(enc.input_ids[0]), tok.convert_ids_to_tokens(lab[0])

def agrupar(toks):
    """Junta subtokens de SentencePiece en palabras (una palabra nueva empieza con '▁')."""
    grupos, palabras_ = [], []
    for k, t in enumerate(toks):
        if t in ('</s>', '<pad>'):
            continue
        if t.startswith('▁') or not grupos:
            grupos.append([k]); palabras_.append(t.lstrip('▁'))
        else:
            grupos[-1].append(k); palabras_[-1] += t
    return grupos, palabras_

def a_palabras(A, src, tgt):
    gs, ws = agrupar(src); gt, wt = agrupar(tgt)
    W = np.array([[A[np.ix_(r, c)].sum(1).mean() for c in gs] for r in gt])  # filas: promedio; columnas: suma
    return W, ws, wt

In [ ]:
# Caso de estudio: el ejemplo de la Fase 1 (puesto de comida con estrella Michelin), resumen del mT5 sin ajuste
mich = test_s.index[test_s['title'].str.contains('Michelin')]
i_at = int(mich[0]) if len(mich) else 3
A, src, tgt = atencion_cruzada(mt5_att, TEXTOS[i_at], pre_preds[i_at])
W, ws, wt = a_palabras(A, src, tgt)

cols_at = sorted({int(j) for k in range(len(wt)) for j in np.argsort(-W[k])[:2]})   # las 2 más atendidas por fila
fig, ax = plt.subplots(figsize=(13, 6.5))
im = ax.imshow(W[:, cols_at], cmap='Blues', aspect='auto')
ax.set_yticks(range(len(wt)), wt, fontsize=8)
ax.set_xticks(range(len(cols_at)), [f'{ws[j]} ({j+1})' for j in cols_at], rotation=90, fontsize=8)
ax.set_xlabel('palabras del artículo (número de palabra entre paréntesis)', color=TINTA)
ax.set_ylabel('palabras del resumen', color=TINTA)
ax.set_title('Gráfica 7. Atención cruzada: a qué palabra del artículo mira el modelo al escribir cada palabra del resumen',
             loc='left', fontsize=11)
plt.colorbar(im, ax=ax, fraction=0.02, label='peso de atención')
plt.tight_layout(); plt.show()

pd.set_option('display.max_colwidth', 90)
filas = []
for k, w in enumerate(wt):
    top = np.argsort(-W[k])[:2]
    filas.append({'palabra del resumen': w,
                  'más atendida (n.º)': f'{ws[top[0]]} ({top[0]+1})', 'peso': round(float(W[k, top[0]]), 3),
                  'contexto en el artículo': ' '.join(ws[max(0, top[0]-4): top[0]+5]),
                  'segunda (n.º)': f'{ws[top[1]]} ({top[1]+1})'})
print('Resumen:', pre_preds[i_at])
display(pd.DataFrame(filas))

**Lectura del mapa (gráfica 7).**
- **La diagonal es copia.** Casi todo el resumen sale de una sola frase del artículo: al escribir *dijo a BBC Mundo que todavía no ha decidido dónde colgar*, cada palabra mira justo a la misma palabra en el texto, que dice *"Chan Hon Meng dijo a BBC Mundo que todavía no ha decidido dónde colgar la placa con la estrella Michelin"*. La atención cruzada funciona aquí como un puntero que copia.
- **Corrección a la Fase 1.** En la sección 3.5 dijimos que el modelo inventó que el dueño no ha decidido dónde colgar la estrella. El mapa muestra que esa frase sí está en el artículo: el modelo resumió *"la placa con la estrella"* como *"una estrella"* (al escribir *estrella* mira a *placa* y a las menciones de *estrella*). El dato no es inventado; el problema es que eligió un detalle secundario en lugar del hecho principal (el platillo de menos de US$2 y la estrella Michelin del puesto).
- **El error real es "en Hong Kong".** Al escribir *Hong Kong*, la atención va a esas palabras dentro del nombre del puesto, *Hong Kong Soya Sauce Chicken Rice and Noodle*, pero el puesto está en Singapur. El modelo tomó parte del nombre como si fuera el lugar. Es un error de relación: *Hong* y *Kong* sí aparecen en el artículo, así que el verificador no lo puede detectar (sección 5.3).
- **Las palabras de función** como *El* o *en* tienen atención baja y repartida: el decodificador las produce con su propio conocimiento del español (autoatención), no copiándolas del texto.

In [ ]:
# ¿De qué parte del artículo sale cada palabra del resumen? (sistema final, artículos que llenan los 512 tokens)
# Para cada palabra del resumen se toma el token del artículo con más atención cruzada y se ve en qué tramo cae.
TRAMO, N_TRAMOS = 64, 8
cuenta, n_art = np.zeros(N_TRAMOS), 0
for t, s in zip(TEXTOS, final_preds):
    A, src, _ = atencion_cruzada(mt5_fin_att, t, s)
    if len(src) < 512:              # solo artículos truncados: así todos los tramos existen
        continue
    A = A[:-1, :-1]                 # sin </s> del resumen ni del artículo
    cuenta += np.bincount(A.argmax(1) // TRAMO, minlength=N_TRAMOS)[:N_TRAMOS] / len(A)
    n_art += 1
pct_tramo = 100 * cuenta / max(n_art, 1)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(range(N_TRAMOS), pct_tramo, 0.7, color=COL[FINAL], edgecolor='white', linewidth=1)
for k, v in enumerate(pct_tramo):
    ax.text(k, v + 0.5, f'{v:.0f} %', ha='center', fontsize=8, color=TINTA)
ax.axhline(100 / N_TRAMOS, color=TINTA, ls='--', lw=1, label=f'si saliera de todo el artículo por igual ({100 / N_TRAMOS:.1f} %)')
ax.set_ylim(0, max(pct_tramo.max(), 100 / N_TRAMOS) * 1.35)
ax.legend(frameon=False, fontsize=8, loc='upper right')
ax.set_xticks(range(N_TRAMOS), [f'{k*TRAMO+1}–{(k+1)*TRAMO}' for k in range(N_TRAMOS)], fontsize=8)
ax.set_xlabel('posición en el artículo (tokens)', color=TINTA)
ax.set_ylabel('% de palabras del resumen', color=TINTA)
ax.set_title(f'Gráfica 8. ¿De qué parte del artículo sale cada palabra del resumen? ({n_art} artículos)', loc='left', fontsize=11)
estilo(ax); plt.tight_layout(); plt.show()

pct_inicio = pct_tramo[:2].sum()
if pct_inicio > 35:
    lectura = ('El modelo se apoya sobre todo en el inicio de la nota, como un periodista que resume la entrada. '
               'Por eso perder el final del artículo afecta menos de lo que parece, pero cuando el dato clave está al final '
               'el modelo no lo usa (casos de la sección 5.3).')
else:
    lectura = 'El modelo toma palabras de toda la parte que alcanza a leer, no solo del inicio.'
display(Markdown(f'**Lectura.** En {n_art} artículos de prueba que llenan los 512 tokens, el {pct_inicio:.0f} % de las palabras '
                 f'del resumen tiene su palabra más atendida en los primeros 128 tokens (si salieran de todo el texto por igual '
                 f'sería 25 %) y el {pct_tramo[-2:].sum():.0f} % en los últimos 128. {lectura}'))

### 6.3 Límites encontrados
- **ROUGE no mide fidelidad.** Compara contra un solo resumen que, en 4 de cada 10 casos, trae datos que no están en la noticia. Un sistema que solo usa el artículo no puede llegar a 100, y por eso agregamos la métrica de datos sin respaldo.
- **El verificador es superficial.** Revisa que las cifras y los nombres existan en el artículo, no que estén bien relacionados. El caso de *Hong Kong* (6.2) lo pasa sin problema. También marca datos ciertos que el texto no menciona, como *Donald* en "Donald Trump".
- **Solo lee 512 tokens.** El mT5 casi nunca ve el artículo completo (sección 5.3), y la gráfica 8 muestra de qué parte de esos 512 tokens toma sus palabras.
- **Resúmenes de una oración.** Aunque el límite es de 120 palabras (60 s), el modelo aprendió el estilo de la entrada de BBC Mundo: una oración de unas 20 a 30 palabras (10 a 15 s al aire). Para llenar el minuto habría que entrenar con resúmenes más largos.
- **Dominio.** XL-Sum son notas internacionales de BBC Mundo. El modelo no vio noticias locales como las de Radio Cumbres, con nombres de comunidades y funcionarios que no aparecen en el corpus.
- **Cómputo y muestra.** En GPU tarda menos de un segundo, pero en CPU tarda varios segundos por noticia. La evaluación usa 200 artículos de prueba por tiempo de cómputo; los intervalos de 6.1 indican cuánto puede variar el resultado.

### 6.4 Conclusión final

In [ ]:
TF, TB = tabla_final.loc[FINAL], tabla_final.loc[BASE]
ic_r1 = bootstrap_ic(pa[FINAL]['rouge1'] - pa[BASE]['rouge1'])
mejor_en = [NOMBRE_M[m] for m in METRICAS if TF[m] > TB[m]]
if ic_r1[0] > 0:
    signif = 'El intervalo no incluye el cero, así que la mejora en ROUGE-1 no se explica por la muestra de prueba.'
elif ic_r1[1] < 0:
    signif = 'El intervalo queda completo por debajo de cero: con esta muestra, la línea base es mejor en ROUGE-1.'
else:
    signif = 'El intervalo incluye el cero, así que la diferencia en ROUGE-1 no es concluyente con esta muestra.'
display(Markdown(f"""
La solución con Transformer (**{FINAL}**) {'supera a la línea base TF-IDF en ' + ', '.join(mejor_en) if mejor_en else 'no supera a la línea base TF-IDF en ROUGE'}:
ROUGE-1 {TF.rouge1:.2f} contra {TB.rouge1:.2f} ({TF.rouge1 - TB.rouge1:+.2f} puntos; IC 95 %: {ic_r1[0]:+.2f} a {ic_r1[1]:+.2f}),
ROUGE-2 {TF.rouge2:.2f} contra {TB.rouge2:.2f} y ROUGE-L {TF.rougeL:.2f} contra {TB.rougeL:.2f}.
{signif}
Ambos respetan el límite de 120 palabras en el {TF['% ≤120 palabras']:.0f} % de los casos.

La comparación tiene otra cara: el TF-IDF tarda {TB['s por artículo']:.3f} s por artículo y nunca inventa datos (copia oraciones),
mientras que el mT5 tarda {TF['s por artículo']:.2f} s y puede escribir datos que no están en la noticia. El verificador deja
ese problema en {TF['% con dato no encontrado']:.1f} % de los resúmenes, y la interfaz los marca para que el locutor los revise.

El análisis de atención explica cómo resume el modelo: copia frases del artículo{', sobre todo del inicio,' if pct_inicio > 35 else ''} y algunos de
sus errores más difíciles de detectar no son datos inventados sino datos reales mal relacionados, como ubicar en *Hong Kong* un
puesto que está en Singapur.
"""))

### 6.5 Tabla de resultados para el README (en inglés)

In [ ]:
nombres_en = {BASE: f'Baseline: extractive TF-IDF ({BEST_L} words)', 'mT5 sin ajuste': 'mT5_multilingual_XLSum (no fine-tuning)',
              'mT5 sin ajuste + verificador': 'mT5 (no fine-tuning) + fact checker',
              'mT5 ajustado': 'mT5 fine-tuned on Spanish', 'mT5 ajustado + verificador': 'mT5 fine-tuned + fact checker'}
cols_en = {'rouge1': 'ROUGE-1', 'rouge2': 'ROUGE-2', 'rougeL': 'ROUGE-L', '% ≤120 palabras': '% ≤ 120 words',
           '% con dato no encontrado': '% with unsupported facts', 's por artículo': 'sec / article'}
lineas = ['| System (test, 200 articles) | ' + ' | '.join(cols_en.values()) + ' |', '| --- ' * (len(cols_en) + 1) + '|']
for n, r in tabla_test[list(cols_en)].astype(float).iterrows():
    lineas.append(f'| {nombres_en[n]}{" **(final)**" if n == FINAL else ""} | ' + ' | '.join(f'{v:g}' for v in r) + ' |')
print('\n'.join(lineas))
print(f'\nROUGE-1 gain over the baseline: {TF.rouge1 - TB.rouge1:+.2f} points (95% bootstrap CI {ic_r1[0]:+.2f} to {ic_r1[1]:+.2f})')

---
### Anexo A: abrir la interfaz desde Colab (con GPU)
Clona el repositorio (o sube a `/content` los archivos `app.py`, `resumidor.py` y `palabras_comunes.txt`) y ejecuta la celda. Aparece un enlace público temporal (`*.gradio.live`) que funciona mientras la celda siga corriendo. Con GPU responde en menos de un segundo.

In [ ]:
# !git clone https://github.com/USUARIO/REPO.git /content/repo
# %cd /content/repo
import os
os.environ['MODEL_DIR'] = MODELO_INTERFAZ
!pip -q install gradio
!python app.py --share

### Anexo B: publicar la interfaz en Hugging Face Spaces (enlace permanente y gratuito)
1. Crea una cuenta en huggingface.co y un token con permiso de escritura (*Settings › Access Tokens › New token › Write*).
2. Clona el repositorio en `/content/repo` (primera línea de la celda) y escribe tu usuario en `USUARIO`.
3. Ejecuta la celda: sube el modelo ajustado al Hub (si es el que se eligió), crea el Space, sube la interfaz y le dice qué modelo usar. El Space tarda unos minutos en construirse la primera vez.

In [ ]:
# !git clone https://github.com/USUARIO/REPO.git /content/repo
from huggingface_hub import login, HfApi
login()                                   # pega aquí el token de escritura
USUARIO = 'tu_usuario_de_huggingface'
REPO_MODELO = f'{USUARIO}/mt5-radio-cumbres'
REPO_SPACE  = f'{USUARIO}/news-in-60-seconds'
CARPETA     = Path('/content/repo')

api = HfApi()
if usar_ajustado:                         # el modelo sin ajuste ya está público en el Hub
    api.create_repo(REPO_MODELO, exist_ok=True)
    ft_model.push_to_hub(REPO_MODELO); tok.push_to_hub(REPO_MODELO)
modelo_space = REPO_MODELO if usar_ajustado else MODEL

api.create_repo(REPO_SPACE, repo_type='space', space_sdk='gradio', exist_ok=True)
api.upload_folder(folder_path=str(CARPETA / 'space'), repo_id=REPO_SPACE, repo_type='space')   # README y requirements
for f in ['app.py', 'resumidor.py', 'palabras_comunes.txt']:
    api.upload_file(path_or_fileobj=str(CARPETA / f), path_in_repo=f, repo_id=REPO_SPACE, repo_type='space')
api.add_space_variable(REPO_SPACE, 'MODEL_DIR', modelo_space)
print('Interfaz:', f'https://huggingface.co/spaces/{REPO_SPACE}')
print('Modelo que usa:', modelo_space)